# Samodzielna analiza segmentacyjna - klienci kampanii marketingowej

W tym notebooku przygotujemy dane do analizy skupień klientów.

Zbiór zawiera informacje o klientach, ich cechach demograficznych, zachowaniach zakupowych oraz reakcjach na kampanie marketingowe.

Celem zadania będzie samodzielne wykonanie segmentacji klientów oraz interpretacja otrzymanych grup.

W przeciwieństwie do zbioru Iris nie mamy tutaj jednej „prawdziwej” etykiety grupy.  
Zmienna `Response` może być jednak wykorzystana pomocniczo do sprawdzenia, czy otrzymane segmenty różnią się skłonnością do odpowiedzi na kampanię.

In [ ]:
# Dodatkowe ułatwienia w ustawieniach wyświetlania i obsłudze ostrzeżeń

from IPython.display import display, HTML
import warnings

# Wyłączenie przewijanej ramki wyników w Jupyterze
display(HTML("<style>.output_scroll { height: auto !important; }</style>"))

# Wyłączenie wybranych ostrzeżeń
warnings.simplefilter(action="ignore", category=UserWarning)

In [ ]:
# Upewnijmy się najpierw, jaki jest nasz bieżący folder roboczy

import os
print(os.getcwd())

In [ ]:
# Obsługa zbiorów danych
import pandas as pd
import numpy as np

# Obsługa wykresów
import matplotlib.pyplot as plt
import seaborn as sns

# Uzupełnianie braków danych
from sklearn.impute import SimpleImputer, KNNImputer

# Wartości odstające
from feature_engine.outliers import Winsorizer

# Skalowanie wartości zmiennych
from sklearn.preprocessing import MinMaxScaler

# Pokazuje wszystkie kolumny zbioru podczas podglądu
pd.set_option("display.max_columns", None)

In [ ]:
# Plik marketing_campaign.csv powinien znajdować się w tym samym folderze co notebook.
# Parametr sep=None pozwala automatycznie wykryć separator kolumn.

marketing_df = pd.read_csv(
    "marketing_campaign.csv",
    sep=None,
    engine="python"
)

marketing_df.head()

## Pierwsze pytania do danych

Na początku sprawdzimy strukturę zbioru danych.

Interesuje nas:

- ile obserwacji i zmiennych zawiera zbiór,
- co oznaczają wiersze i kolumny,
- jakie są typy danych,
- które zmienne są identyfikatorami,
- które zmienne mają charakter jakościowy, ilościowy, binarny lub czasowy,
- które zmienne mogą pełnić rolę pomocniczą przy późniejszej interpretacji segmentów.

In [ ]:
# Liczba obserwacji i liczba zmiennych
print(marketing_df.shape)

In [ ]:
# Wyświetl pierwszych 10 rekordów
marketing_df.head(10)

In [ ]:
# Podstawowe sprawdzenie typów danych
marketing_df.dtypes

**Dt_Customer ma typ "object" a jest to zmienna z datą. Zamienimy typ tej zmiennej.**

In [ ]:
# Zamiana typu object na typ datetime dla zmiennej Dt_Customer

marketing_df["Dt_Customer"] = pd.to_datetime(
    marketing_df["Dt_Customer"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Ponowne sprawdzenie typów danych

marketing_df.dtypes

In [ ]:
# Sprawdzenie liczby unikalnych wartości
marketing_df.nunique().sort_values()

In [ ]:
# Identyfikacja zmiennych stałych, czyli takich, które mają tylko jedną unikalną wartość

constant_vars = [
    var for var in marketing_df.columns
    if marketing_df[var].nunique() == 1
]

print(constant_vars)

In [ ]:
# Usunięcie zmiennych stałych

marketing_df = marketing_df.drop(columns=constant_vars)

# Ponowne sprawdzenie liczby unikalnych wartości
marketing_df.nunique().sort_values()

## Ważna decyzja: rola zmiennych w analizie

Przed analizą skupień musimy określić, które zmienne będą używane do tworzenia segmentów, a które zostaną wykorzystane tylko pomocniczo.

W tym zbiorze:

- `ID` jest identyfikatorem klienta,
- `Response` będzie zmienną pomocniczą do interpretacji segmentów,
- `Year_Birth` i `Dt_Customer` wymagają przekształcenia na bardziej użyteczne zmienne,
- zmienne tekstowe trzeba będzie przekodować,
- zmienne binarne 0/1 można potraktować jako osobny typ informacji o zachowaniu klienta.

In [ ]:
# Zmienna identyfikująca klienta
id_vars = ["ID"]

print(id_vars)

In [ ]:
# Zmienna pomocnicza
# Nie będzie używana do budowy segmentów, ale może pomóc w ich interpretacji
additional = ["Response"]

print(additional)

In [ ]:
# Zmienne jakościowe tekstowe

categorical_vars = [
    var for var in marketing_df.columns
    if marketing_df[var].dtype == "object"
]

print(categorical_vars)

In [ ]:
# Zmienne binarne 0/1
# Nie uwzględniamy ID ani zmiennej Response

binary_vars = [
    var for var in marketing_df.columns
    if marketing_df[var].nunique() == 2
    and var not in id_vars
    and var not in additional
    and marketing_df[var].dtype != "object"
]

print(binary_vars)

In [ ]:
# Zmienne czasowe lub wymagające przekształcenia czasowego
temporal_vars = ["Year_Birth", "Dt_Customer"]

print(temporal_vars)

In [ ]:
# Zmienne ilościowe

num_vars = [
    var for var in marketing_df.columns
    if var not in id_vars
    and var not in additional
    and var not in categorical_vars
    and var not in binary_vars
    and var not in temporal_vars
]

print(num_vars)

## Ocena jakości zbioru danych

Przed przygotowaniem danych do segmentacji sprawdzimy ich jakość.

Na tym etapie ocenimy:

- braki danych,
- rozkłady zmiennych ilościowych,
- potencjalne wartości odstające,
- liczności kategorii w zmiennych jakościowych,
- rozkłady zmiennych binarnych.

W analizie segmentacyjnej jest to szczególnie ważne, ponieważ wartości odstające, braki danych oraz bardzo rzadkie kategorie mogą wpływać na wynik grupowania klientów.

In [ ]:
# Raport braków danych

missing_report = pd.DataFrame({
    "Liczba braków": marketing_df.isnull().sum(),
    "Procent braków": (marketing_df.isnull().mean() * 100).round(2)
})

missing_report.sort_values("Procent braków", ascending=False)

## Rozkłady zmiennych ilościowych

Najpierw sprawdzimy zmienne ilościowe, takie jak dochód, liczba zakupów, wydatki w różnych kategoriach oraz liczba wizyt na stronie.

Interesuje nas:

- zakres wartości,
- skośność rozkładów,
- potencjalne wartości odstające,
- zmienne o bardzo nietypowych rozkładach.

In [ ]:
# Histogramy dla zmiennych ilościowych

marketing_df[num_vars].hist(
    bins=20,
    figsize=(15, 15)
)

plt.suptitle("Rozkłady zmiennych ilościowych", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def plot_box_strip(df, variables, hue=None):
    """
    Tworzy wykresy pudełkowe z nałożonymi punktami obserwacji dla podanych zmiennych.

    Parametry:
    - df: DataFrame zawierający dane,
    - variables: lista nazw kolumn ze zmiennymi ilościowymi,
    - hue: opcjonalna zmienna grupująca.
    """

    for var in variables:
        plt.figure(figsize=(8, 6))

        sns.boxplot(
            data=df,
            y=var,
            x=hue,
            fliersize=0,
            color="green"
        )

        sns.stripplot(
            data=df,
            y=var,
            x=hue,
            jitter=0.1,
            alpha=0.3,
            color="black"
        )

        plt.title(f"Wykres pudełkowy z obserwacjami: {var}")
        plt.xlabel(hue if hue else "")
        plt.ylabel(var)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_box_strip(marketing_df, num_vars)

In [ ]:
# Podsumowanie podstawowych statystyk opisowych
marketing_df[num_vars].describe().T

Wartości odstające w danych marketingowych nie muszą oznaczać błędów.

Klient o bardzo wysokich wydatkach, dużej liczbie zakupów albo bardzo wysokim dochodzie może być rzeczywiście nietypowy, ale jednocześnie bardzo ważny z perspektywy segmentacji.

Dlatego wartości odstające należy najpierw zidentyfikować, a dopiero później zdecydować, czy wymagają przycięcia, usunięcia czy pozostawienia w analizie.

## Rozkłady zmiennych jakościowych

Następnie sprawdzimy liczności kategorii w zmiennych jakościowych.

Jest to ważne, ponieważ bardzo rzadkie kategorie mogą utrudniać kodowanie zmiennych i późniejszą interpretację segmentów.

In [ ]:
# Wykresy słupkowe dla zmiennych jakościowych

for var in categorical_vars:
    plt.figure(figsize=(6, 4))

    marketing_df[var].value_counts().plot(
        kind="bar",
        color="skyblue",
        edgecolor="black"
    )

    plt.title(f"Liczność kategorii dla zmiennej: {var}")
    plt.xlabel(var)
    plt.ylabel("Liczność")
    plt.xticks(rotation=45)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

## Rozkłady zmiennych binarnych

Zmienne binarne 0/1 opisują m.in. wcześniejsze reakcje klienta na kampanie oraz wystąpienie reklamacji.

Warto sprawdzić, czy w tych zmiennych występują bardzo rzadkie wartości, ponieważ mogą one mieć znaczenie przy późniejszej interpretacji segmentów.

In [ ]:
# Liczności wartości 0/1 dla zmiennych binarnych

for var in binary_vars:
    display(
        pd.DataFrame({
            "Liczność": marketing_df[var].value_counts(dropna=False),
            "Procent": (marketing_df[var].value_counts(dropna=False, normalize=True) * 100).round(2)
        })
    )

## Czyszczenie i przekształcanie danych

Po wstępnym rozpoznaniu danych przechodzimy do ich przygotowania do analizy segmentacyjnej.

W tym kroku:

- uzupełnimy braki danych,
- ograniczymy wpływ wartości skrajnych,
- przekształcimy wybrane zmienne jakościowe do postaci liczbowej,
- utworzymy zmienne syntetyczne opisujące wcześniejsze reakcje na kampanie,
- przygotujemy końcowy zestaw zmiennych do analizy skupień.

In [ ]:
# Uzupełnianie braków danych w zmiennej Income za pomocą mediany

simple_imputer = SimpleImputer(strategy="median")

marketing_df.loc[:, ["Income"]] = simple_imputer.fit_transform(
    marketing_df[["Income"]]
)

# Sprawdzenie, czy braki zostały uzupełnione

marketing_df[["Income"]].isnull().sum()

## Wartości odstające

W danych marketingowych wartości odstające nie zawsze są błędami.  
Klient o bardzo wysokim dochodzie lub bardzo dużych wydatkach może być nietypowy, ale nadal ważny z punktu widzenia segmentacji.

W tym przykładzie usuniemy jedynie skrajnie nietypowy przypadek dla zmiennej `Income`, a następnie zastosujemy winsoryzację dla zmiennych ilościowych.

In [ ]:
# Zachowujemy kopię danych przed usunięciem obserwacji odstających

marketing_df_before_outlier_filtering = marketing_df.copy()

# Usunięcie przypadków, gdzie Income > 300000

marketing_df = marketing_df.loc[marketing_df["Income"] <= 300000].copy()

marketing_df.shape

W praktycznej analizie warto dodatkowo rozważyć, czy inne bardzo nietypowe przypadki również powinny zostać usunięte lub potraktowane osobno.

Nie należy jednak usuwać obserwacji tylko dlatego, że są nietypowe. Decyzja powinna mieć uzasadnienie merytoryczne.

In [ ]:
# Przed winsoryzacją zamieniamy zmienne ilościowe na typ float.
# Winsoryzacja może tworzyć wartości niecałkowite, ponieważ granice przycinania
# są obliczane na podstawie kwartylów.

marketing_df[num_vars] = marketing_df[num_vars].astype(float)

# capping_method='iqr' - metoda oparta na rozstępie kwartylowym
# fold=1.5 - parametr sterujący szerokością granic przycinania
# tail='both' - przycinanie wartości odstających z obu stron rozkładu

winsorizer = Winsorizer(
    capping_method="iqr",
    fold=1.5,
    tail="both"
)

marketing_df.loc[:, num_vars] = winsorizer.fit_transform(marketing_df[num_vars])

In [ ]:
# Sprawdzenie efektu winsoryzacji
plot_box_strip(marketing_df, num_vars)

### Wróćmy do analizy zmiennych jakościowych 

In [ ]:
# Wykresy słupkowe dla zmiennych jakościowych

for var in categorical_vars:
    plt.figure(figsize=(6, 4))

    marketing_df[var].value_counts().plot(
        kind="bar",
        color="skyblue",
        edgecolor="black"
    )

    plt.title(f"Liczność kategorii dla zmiennej: {var}")
    plt.xlabel(var)
    plt.ylabel("Liczność")
    plt.xticks(rotation=45)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

## Przekształcenie zmiennej `Education`

Zmienną `Education` przekształcimy do postaci binarnej.

Nowa zmienna `IsAdvancedDegree` przyjmie wartość:

- `1` dla klientów z wyższym poziomem wykształcenia,
- `0` dla pozostałych klientów.

To uproszczenie ułatwi późniejsze wykorzystanie tej informacji w analizie skupień.

In [ ]:
marketing_df["IsAdvancedDegree"] = marketing_df["Education"].isin(
    ["PhD", "Master", "2n Cycle"]
).astype(int)

marketing_df[["Education", "IsAdvancedDegree"]].head()

## Przekształcenie zmiennej `Marital_Status`

Zmienną `Marital_Status` przekształcimy do postaci binarnej.

Nowa zmienna `IsTogether` przyjmie wartość:

- `1` dla osób pozostających w związku,
- `0` dla pozostałych osób.

Kategorie o małej liczności zostaną włączone do szerszej grupy `0`.

In [ ]:
marketing_df["IsTogether"] = marketing_df["Marital_Status"].isin(
    ["Married", "Together"]
).astype(int)

marketing_df[["Marital_Status", "IsTogether"]].head()

## Przekształcenie zmiennych kampanijnych

Zmienne `AcceptedCmp1`–`AcceptedCmp5` informują, czy klient zaakceptował wcześniejsze kampanie.

Ponieważ akceptacja pojedynczych kampanii jest stosunkowo rzadka, utworzymy zmienną syntetyczną `TotalAcceptedCmp`, która będzie oznaczać liczbę zaakceptowanych kampanii.

Taka zmienna jest łatwiejsza do interpretacji niż pięć osobnych, rzadkich zmiennych binarnych.

In [ ]:
campaign_vars = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5"
]

marketing_df["TotalAcceptedCmp"] = marketing_df[campaign_vars].sum(axis=1)

marketing_df[["TotalAcceptedCmp"]].hist(bins=10, figsize=(6, 4))

plt.title("Liczba zaakceptowanych kampanii")
plt.xlabel("Liczba kampanii")
plt.ylabel("Liczność")
plt.tight_layout()
plt.show()

**Niewielka liczba osób akceptujących powyżej jednej kampainii**

In [ ]:
# Tworzenie zmiennej 0-1: czy zaakceptowano przynajmniej jedną kampanię
marketing_df["AcceptedAnyCmp"] = (marketing_df["TotalAcceptedCmp"] > 0).astype(int)

marketing_df[["AcceptedAnyCmp"]].hist(bins=2, figsize=(5, 4))

plt.title("Czy zaakceptowano przynajmniej jedną kampanię")
plt.xlabel("AcceptedAnyCmp")
plt.ylabel("Liczność")
plt.tight_layout()
plt.show()

## Zmienna `Complain`

Zmienna `Complain` jest bardzo rzadka, dlatego nie będziemy jej wykorzystywać jako zmiennej segmentacyjnej.

Można ją jednak ewentualnie zachować jako zmienną pomocniczą do późniejszego opisu segmentów.

In [ ]:
# Zmienne, których nie używamy bezpośrednio do budowy segmentów

excluded_vars = (
    id_vars
    + additional
    + categorical_vars
    + temporal_vars
    + campaign_vars
    + ["Complain", "AcceptedAnyCmp"]
)

# Zmienne używane do segmentacji

predictors = [
    var for var in marketing_df.columns
    if var not in excluded_vars
    and pd.api.types.is_numeric_dtype(marketing_df[var])
]

print(predictors)

## Przekształcenie zmiennych czasowych

Zmiennych czasowych zwykle nie wykorzystujemy w analizie skupień bezpośrednio.  
Zamiast tego tworzymy z nich bardziej interpretowalne cechy.

W tym przypadku wykorzystamy:

- `Year_Birth` do obliczenia przybliżonego wieku klienta,
- `Dt_Customer` do obliczenia stażu klienta, czyli liczby dni od momentu rejestracji.

Jako datę odniesienia przyjmujemy 1 stycznia 2015 roku.

In [ ]:
marketing_df[temporal_vars].hist(bins=30, figsize=(15,15))
plt.show()

### Obliczymy wiek klienta oraz jak długo w dniach jest klientem sklepu - zakładamy że mamy 1 stycznia 2015

In [ ]:
# Definiujemy datę odniesienia

reference_date = pd.Timestamp("2015-01-01")

# Obliczenie przybliżonego wieku klienta
# Uwaga: jest to uproszczenie, ponieważ dysponujemy tylko rokiem urodzenia.

marketing_df["Age"] = reference_date.year - marketing_df["Year_Birth"]

marketing_df[["Age"]].hist(bins=30, figsize=(8, 5))

plt.xlabel("Wiek")
plt.ylabel("Liczność")
plt.title("Rozkład wieku klientów")
plt.tight_layout()
plt.show()

In [ ]:
# Usunięcie przypadków, gdzie Age > 100

marketing_df = marketing_df.loc[marketing_df["Age"] <= 100].copy()

marketing_df[["Age"]].hist(bins=30, figsize=(8, 5))

plt.xlabel("Wiek")
plt.ylabel("Liczność")
plt.title("Rozkład wieku klientów po usunięciu wartości skrajnych")
plt.tight_layout()
plt.show()

In [ ]:
# Obliczenie stażu klienta w dniach

marketing_df["CustomerTenure"] = (
    reference_date - marketing_df["Dt_Customer"]
).dt.days

marketing_df[["CustomerTenure"]].hist(bins=30, figsize=(8, 5))

plt.xlabel("Staż klienta w dniach")
plt.ylabel("Liczność")
plt.title("Rozkład stażu klienta")
plt.tight_layout()
plt.show()

## Końcowy zestaw zmiennych do segmentacji

Po przekształceniach tworzymy końcową listę zmiennych, które zostaną wykorzystane do analizy skupień.

Do modelu nie włączamy bezpośrednio zmiennych `Year_Birth` i `Dt_Customer`, ponieważ zostały zastąpione przez bardziej interpretowalne zmienne:

- `Age`,
- `CustomerTenure`.

In [ ]:
derived_vars = [
    "CustomerTenure",
    "Age",
    "TotalAcceptedCmp",
    "IsAdvancedDegree",
    "IsTogether"
]

print(derived_vars)

In [ ]:
predictors = num_vars + derived_vars

print(predictors)

## Sprawdźmy powiązania pomiędzy zmiennymi

- Korelacje między zmiennymi — za pomocą korelacji Spearmana
- Graficzne przedstawienie za pomocą mapy ciepła

In [ ]:
num_vars_new = [var for var in predictors if marketing_df[var].nunique() > 2]

# Obliczanie macierzy korelacji rang Spearmana
spearman_corr_matrix = marketing_df[num_vars_new].corr(method='spearman')

spearman_corr_matrix.head(20)

In [ ]:
# Tworzenie mapy ciepła
plt.figure(figsize=(12, 10))  # Możesz dostosować wymiary
sns.heatmap(spearman_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

# Tytuł wykresu
plt.title("Spearman Rank Correlation Heatmap")

# Wyświetlenie wykresu
plt.show()

### Analiza skupień wymaga przeskalowania danych do wspólnego zakresu
1. Wynik skalowania umieścimy w nowej ramce danych
2. Zobaczymy jak wyglądają rozkłady zmiennych po skalowaniu

In [ ]:
# Skalujemy wartości predyktorów
# Przed wykonaniem skalowania wykonamy kopię ramki danych 
scaled_marketing_df = marketing_df.copy()

min_max_scaler = MinMaxScaler()
scaled_marketing_df[predictors]= min_max_scaler.fit_transform(scaled_marketing_df[predictors])

In [ ]:
scaled_marketing_df[predictors].head()

In [ ]:
plot_box_strip(scaled_marketing_df, predictors)

# Zadanie 5 - samodzielna segmentacja klientów

Dane zostały przygotowane do analizy skupień.

Na podstawie przygotowanych ramek:

- `marketing_df`,
- `scaled_marketing_df`,
- `predictors`

wykonaj samodzielną analizę segmentacyjną klientów.

Wykorzystaj wcześniejsze notebooki jako wzorzec postępowania.

## Co należy zrobić?

1. Dobierz liczbę skupień, korzystając z co najmniej dwóch kryteriów, np.:
   - wykresu osypiska,
   - wskaźnika sylwetki,
   - Calinski-Harabasz Score,
   - Davies-Bouldin Score.

2. Zbuduj co najmniej jeden model segmentacyjny, np.:
   - KMeans,
   - aglomeracyjną analizę skupień,
   - GMM,
   - DBSCAN.

3. Oceń jakość otrzymanych segmentów:
   - sprawdź liczności segmentów,
   - oblicz wybrane miary jakości,
   - przygotuj wizualizację segmentów za pomocą PCA lub t-SNE.

4. Zinterpretuj otrzymane segmenty:
   - czym różnią się grupy klientów,
   - które zmienne najlepiej opisują poszczególne segmenty,
   - jak można nazwać otrzymane segmenty.

5. Wykorzystaj zmienną `Response` jako zmienną pomocniczą:
   - sprawdź, czy segmenty różnią się udziałem klientów, którzy odpowiedzieli na kampanię,
   - przygotuj tabelę lub wykres pokazujący odsetek odpowiedzi w segmentach.